# 01 — Grid sanity check

Sanity-check the foundation layer (`config`, `grid`, `utils`). **No physics, no animation** — only the geometry.

If every cell here runs cleanly and the plots look right, the foundation is locked. The physics solver (Rafie) and visualiser (teammate) can then be built on top without touching this layer.

In [ ]:
import sys, pathlib

# Notebook lives in notebook/, source lives in src/ — make ../src importable.
SRC = pathlib.Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

from config import MATERIALS
from grid   import RoomGrid
from utils  import (
    coord_to_cell, cell_to_coord,
    x_lines, y_lines,
    boundary_mask, interior_mask, wall_cells, WALL_SIDES,
    empty_field,
)

## Constructing the grid

`dx`, `dt`, `NX`, `NY` are all **derived** from the physical constants in `config.py` (speed of sound, source frequency, points-per-wavelength, CFL factor). Nothing is hand-tuned.

In [ ]:
grid = RoomGrid(alpha=MATERIALS["Wood"])
print(grid)

field = empty_field(grid)
print(f"Pressure field shape: {field.shape}, dtype: {field.dtype}")

## Coordinate round-trip

`coord_to_cell` rounds to the nearest cell and **clamps** to the grid bounds, so out-of-bounds points map onto the wall (which is what the FDTD step expects).

In [ ]:
test_points = [(0.0, 0.0), (4.0, 3.0), (8.0, 6.0), (10.0, -1.0)]

print(f"{'(x, y) m':>16}  {'(i, j)':>10}  {'(x_back, y_back) m':>22}")
for x, y in test_points:
    i, j = coord_to_cell(x, y, grid)
    xb, yb = cell_to_coord(i, j, grid)
    print(f"{(x, y)!s:>16}  {(i, j)!s:>10}  ({xb:.4f}, {yb:.4f})")

## The empty room

Walls drawn as a thick black rectangle, every 10th grid line drawn faintly so the plot stays readable at high `PPW`.

In [ ]:
xs = x_lines(grid)
ys = y_lines(grid)

fig, ax = plt.subplots(figsize=(8, 6))

# Faint grid lines (every 10th, otherwise the plot is solid grey)
for x in xs[::10]:
    ax.axvline(x, color="0.85", lw=0.5, zorder=0)
for y in ys[::10]:
    ax.axhline(y, color="0.85", lw=0.5, zorder=0)

# Walls
ax.add_patch(plt.Rectangle((0, 0), grid.W, grid.H, fill=False, edgecolor="black", lw=3, zorder=2))

ax.set_xlim(-0.3, grid.W + 0.3)
ax.set_ylim(-0.3, grid.H + 0.3)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(f"Room {grid.W} m × {grid.H} m  |  {grid.NX} × {grid.NY} cells  |  α = {grid.alpha}")
plt.show()

## Boundary mask

These are the cells the FDTD update will **not** write into — pressure there is fixed by the wall boundary condition (modulated by `α`). The interior is everything else.

In [ ]:
bm = boundary_mask(grid)
im = interior_mask(grid)

print(f"Boundary cells : {bm.sum()}")
print(f"Interior cells : {im.sum()}")
print(f"Total          : {bm.size}  (= NX*NY = {grid.NX * grid.NY})")

# Per-wall counts
for side in WALL_SIDES:
    i, j = wall_cells(grid, side)
    print(f"  Wall {side}: {len(i)} cells")

# imshow expects shape (rows, cols) = (y, x), so transpose
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(
    bm.T,
    origin="lower",
    extent=(0, grid.W, 0, grid.H),
    cmap="binary",
)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("boundary_mask — black = wall cell, white = interior")
plt.show()

## Switching wall material

`grid.set_material(name)` looks up the absorption coefficient `α` from the `MATERIALS` preset dict in `config.py`. Cycling through them confirms the material switch works — the FDTD step will read `grid.alpha` each frame, so this is enough to drive a UI dropdown later.

In [ ]:
for name in MATERIALS:
    grid.set_material(name)
    print(f"{name:>14}  →  α = {grid.alpha:.2f}   |   {grid!r}")

## Next steps

- **Physics solver** (Rafie) — inject a source signal into the interior cells, step the 2D scalar wave equation, apply the wall BC using `boundary_mask` + `grid.alpha`.
- **Visualiser** (teammate) — replace the static `imshow` above with an animation of the pressure field over time, plus the source/receiver markers and material dropdown.

Both modules import only from `config`, `grid`, and `utils` — never from each other.